In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

In [3]:
df = pd.read_csv("../data/processed/customer_sales_for_feature_engineering.csv")

df["invoice_date"] = pd.to_datetime(df["invoice_date"])
df["date"] = pd.to_datetime(df["date"])

print("Data shape:", df.shape)
df.head()

Data shape: (392692, 18)


,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country,customer_id_missing,is_cancelled,total_amount,date,year,month,day,day_of_week,hour,year_month
0,536365,85123A,white hanging heart t-light holder,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,0,0,15.30,2010-12-01,2010,12,1,Wednesday,8,2010-12
1,536365,71053,white metal lantern,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,0,0,20.34,2010-12-01,2010,12,1,Wednesday,8,2010-12
2,536365,84406B,cream cupid hearts coat hanger,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,0,0,22.00,2010-12-01,2010,12,1,Wednesday,8,2010-12
3,536365,84029G,knitted union flag hot water bottle,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,0,0,20.34,2010-12-01,2010,12,1,Wednesday,8,2010-12
4,536365,84029E,red woolly hottie white heart.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,0,0,20.34,2010-12-01,2010,12,1,Wednesday,8,2010-12


In [4]:
print("Missing values:")
print(df.isnull().sum())

print("\nUnique invoices:", df["invoice_no"].nunique())
print("Unique customers:", df["customer_id"].nunique())
print("Unique products:", df["stock_code"].nunique())

print("\nDate range:")
print(df["invoice_date"].min(), "to", df["invoice_date"].max())

Missing values:
invoice_no             0
stock_code             0
description            0
quantity               0
invoice_date           0
unit_price             0
customer_id            0
country                0
customer_id_missing    0
is_cancelled           0
total_amount           0
date                   0
year                   0
month                  0
day                    0
day_of_week            0
hour                   0
year_month             0
dtype: int64

Unique invoices: 18532
Unique customers: 4338
Unique products: 3665

Date range:
2010-12-01 08:26:00 to 2011-12-09 12:50:00


In [5]:
product_counts = df["stock_code"].value_counts()

valid_products = product_counts[product_counts >= 10].index

rec_df = df[df["stock_code"].isin(valid_products)].copy()

print("Original shape:", df.shape)
print("Filtered shape:", rec_df.shape)
print("Products kept:", rec_df["stock_code"].nunique())

Original shape: (392692, 18)
Filtered shape: (389480, 18)
Products kept: 2869


In [6]:
rec_df["product_name"] = (
    rec_df["stock_code"].astype(str) 
    + " - " 
    + rec_df["description"].astype(str)
)

rec_df[["stock_code", "description", "product_name"]].head()

,stock_code,description,product_name
0,85123A,white hanging heart t-light holder,85123A - white hanging heart t-light holder
1,71053,white metal lantern,71053 - white metal lantern
2,84406B,cream cupid hearts coat hanger,84406B - cream cupid hearts coat hanger
3,84029G,knitted union flag hot water bottle,84029G - knitted union flag hot water bottle
4,84029E,red woolly hottie white heart.,84029E - red woolly hottie white heart.


In [8]:
basket = (
    rec_df.groupby(["invoice_no", "product_name"])["quantity"]
    .sum()
    .unstack()
    .fillna(0)
)

basket = (basket > 0).astype(int)

print("Basket shape:", basket.shape)

basket.head()

Basket shape: (18504, 3094)


product_name,10002 - inflatable political globe,10080 - groovy cactus inflatable,10120 - doggy rubber,10125 - mini funky design tapes,10133 - colouring pencils brown tube,10135 - colouring pencils brown tube,11001 - asstd design racing car pen,15030 - fan black frame,15034 - paper pocket traveling fan,15036 - assorted colours silk fan,...,90201C - red enamel flower ring,90209B - green enamel+glass hair comb,90209C - pink enamel+glass hair comb,"90214A - letter ""a"" bling key ring","90214K - letter ""k"" bling key ring",BANK CHARGES - bank charges,C2 - carriage,DOT - dotcom postage,M - manual,POST - postage
invoice_no,,,,,,,,,,,,,,,,,,,,,
536365,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536366,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536367,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536368,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536369,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [9]:
from mlxtend.frequent_patterns import apriori, association_rules

In [10]:
frequent_itemsets = apriori(
    basket.astype(bool),
    min_support=0.01,
    use_colnames=True
)

frequent_itemsets = frequent_itemsets.sort_values(
    by="support",
    ascending=False
)

print("Frequent itemsets:", frequent_itemsets.shape)

frequent_itemsets.head(10)

Frequent itemsets: (972, 2)


,support,itemsets
614,0.106518,frozenset({85123A - white hanging heart t-ligh...
236,0.092034,frozenset({22423 - regency cakestand 3 tier})
611,0.086468,frozenset({85099B - jumbo bag red retrospot})
539,0.074524,frozenset({47566 - party bunting})
587,0.074308,frozenset({84879 - assorted colour bird orname...
18,0.069607,frozenset({20725 - lunch bag red retrospot})
325,0.061933,frozenset({22720 - set of 3 cake tins pantry d...
618,0.059393,frozenset({POST - postage})
20,0.056853,frozenset({20727 - lunch bag black skull.})
62,0.055610,frozenset({21212 - pack of 72 retrospot cake c...


In [11]:
rules = association_rules(
    frequent_itemsets,
    metric="lift",
    min_threshold=1.0
)

rules = rules.sort_values(
    by=["lift", "confidence"],
    ascending=False
)

print("Association rules:", rules.shape)

rules.head(10)

Association rules: (934, 14)


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
693,frozenset({23171 - regency tea plate green}),frozenset({23172 - regency tea plate pink}),0.014591,0.012105,0.010917,0.748148,61.802381,1.0,0.010740,3.922522,0.998387,0.691781,0.745062,0.824967
692,frozenset({23172 - regency tea plate pink}),frozenset({23171 - regency tea plate green}),0.012105,0.014591,0.010917,0.901786,61.802381,1.0,0.010740,10.033251,0.995875,0.691781,0.900331,0.824967
926,frozenset({22746 - poppy's playhouse livingroom}),"frozenset({22745 - poppy's playhouse bedroom, ...",0.013619,0.013727,0.010052,0.738095,53.770529,1.0,0.009865,3.765771,0.994952,0.581250,0.734450,0.735189
923,"frozenset({22745 - poppy's playhouse bedroom, ...",frozenset({22746 - poppy's playhouse livingroom}),0.013727,0.013619,0.010052,0.732283,53.770529,1.0,0.009865,3.684424,0.995061,0.581250,0.728587,0.735189
648,frozenset({23175 - regency milk jug pink}),frozenset({23174 - regency sugar bowl green}),0.014700,0.014483,0.011133,0.757353,52.291264,1.0,0.010920,4.061523,0.995510,0.616766,0.753787,0.763005
649,frozenset({23174 - regency sugar bowl green}),frozenset({23175 - regency milk jug pink}),0.014483,0.014700,0.011133,0.768657,52.291264,1.0,0.010920,4.259041,0.995292,0.616766,0.765205,0.763005
924,frozenset({22746 - poppy's playhouse livingroo...,frozenset({22745 - poppy's playhouse bedroom}),0.011619,0.017077,0.010052,0.865116,50.658581,1.0,0.009853,7.287185,0.991784,0.539130,0.862773,0.726862
925,frozenset({22745 - poppy's playhouse bedroom}),frozenset({22746 - poppy's playhouse livingroo...,0.017077,0.011619,0.010052,0.588608,50.658581,1.0,0.009853,2.402526,0.997291,0.539130,0.583771,0.726862
759,frozenset({23170 - regency tea plate roses}),frozenset({23172 - regency tea plate pink}),0.017726,0.012105,0.010646,0.600610,49.614656,1.0,0.010432,2.473507,0.997527,0.554930,0.595716,0.740037
758,frozenset({23172 - regency tea plate pink}),frozenset({23170 - regency tea plate roses}),0.012105,0.017726,0.010646,0.879464,49.614656,1.0,0.010432,8.149237,0.991852,0.554930,0.877289,0.740037


In [12]:
rules_clean = rules.copy()

rules_clean["antecedents"] = rules_clean["antecedents"].apply(
    lambda x: ", ".join(list(x))
)

rules_clean["consequents"] = rules_clean["consequents"].apply(
    lambda x: ", ".join(list(x))
)

rules_clean = rules_clean[
    [
        "antecedents",
        "consequents",
        "support",
        "confidence",
        "lift"
    ]
]

rules_clean.head(20)

,antecedents,consequents,support,confidence,lift
693,23171 - regency tea plate green,23172 - regency tea plate pink,0.010917,0.748148,61.802381
692,23172 - regency tea plate pink,23171 - regency tea plate green,0.010917,0.901786,61.802381
926,22746 - poppy's playhouse livingroom,"22745 - poppy's playhouse bedroom, 22748 - pop...",0.010052,0.738095,53.770529
923,"22745 - poppy's playhouse bedroom, 22748 - pop...",22746 - poppy's playhouse livingroom,0.010052,0.732283,53.770529
648,23175 - regency milk jug pink,23174 - regency sugar bowl green,0.011133,0.757353,52.291264
649,23174 - regency sugar bowl green,23175 - regency milk jug pink,0.011133,0.768657,52.291264
924,"22746 - poppy's playhouse livingroom, 22748 - ...",22745 - poppy's playhouse bedroom,0.010052,0.865116,50.658581
925,22745 - poppy's playhouse bedroom,"22746 - poppy's playhouse livingroom, 22748 - ...",0.010052,0.588608,50.658581
759,23170 - regency tea plate roses,23172 - regency tea plate pink,0.010646,0.600610,49.614656
758,23172 - regency tea plate pink,23170 - regency tea plate roses,0.010646,0.879464,49.614656


In [13]:
strong_rules = rules_clean[
    (rules_clean["confidence"] >= 0.3) &
    (rules_clean["lift"] >= 2.0)
].copy()

strong_rules = strong_rules.sort_values(
    by=["lift", "confidence"],
    ascending=False
)

print("Strong rules:", strong_rules.shape)

strong_rules.head(20)

Strong rules: (612, 5)


,antecedents,consequents,support,confidence,lift
693,23171 - regency tea plate green,23172 - regency tea plate pink,0.010917,0.748148,61.802381
692,23172 - regency tea plate pink,23171 - regency tea plate green,0.010917,0.901786,61.802381
926,22746 - poppy's playhouse livingroom,"22745 - poppy's playhouse bedroom, 22748 - pop...",0.010052,0.738095,53.770529
923,"22745 - poppy's playhouse bedroom, 22748 - pop...",22746 - poppy's playhouse livingroom,0.010052,0.732283,53.770529
648,23175 - regency milk jug pink,23174 - regency sugar bowl green,0.011133,0.757353,52.291264
649,23174 - regency sugar bowl green,23175 - regency milk jug pink,0.011133,0.768657,52.291264
924,"22746 - poppy's playhouse livingroom, 22748 - ...",22745 - poppy's playhouse bedroom,0.010052,0.865116,50.658581
925,22745 - poppy's playhouse bedroom,"22746 - poppy's playhouse livingroom, 22748 - ...",0.010052,0.588608,50.658581
759,23170 - regency tea plate roses,23172 - regency tea plate pink,0.010646,0.600610,49.614656
758,23172 - regency tea plate pink,23170 - regency tea plate roses,0.010646,0.879464,49.614656


In [14]:
frequent_itemsets.to_csv(
    "../data/processed/recommendation_frequent_itemsets.csv",
    index=False
)

rules_clean.to_csv(
    "../data/processed/recommendation_association_rules.csv",
    index=False
)

strong_rules.to_csv(
    "../data/processed/recommendation_strong_rules.csv",
    index=False
)

print("Apriori recommendation outputs saved successfully.")

Apriori recommendation outputs saved successfully.


In [15]:
def recommend_from_rules(product_name, rules_df, top_n=5):
    matched_rules = rules_df[
        rules_df["antecedents"].str.contains(product_name, case=False, regex=False)
    ].copy()

    if matched_rules.empty:
        return pd.DataFrame(columns=["antecedents", "consequents", "confidence", "lift"])

    recommendations = matched_rules.sort_values(
        by=["confidence", "lift"],
        ascending=False
    ).head(top_n)

    return recommendations[
        ["antecedents", "consequents", "confidence", "lift"]
    ]

In [16]:
sample_product = "regency tea plate green"

recommend_from_rules(
    sample_product,
    strong_rules,
    top_n=5
)

,antecedents,consequents,confidence,lift
474,23171 - regency tea plate green,23170 - regency tea plate roses,0.848148,47.847967
693,23171 - regency tea plate green,23172 - regency tea plate pink,0.748148,61.802381


In [17]:
customer_product_matrix = (
    rec_df.groupby(["customer_id", "product_name"])["quantity"]
    .sum()
    .unstack()
    .fillna(0)
)

customer_product_matrix = (customer_product_matrix > 0).astype(int)

print("Customer-product matrix shape:", customer_product_matrix.shape)

customer_product_matrix.head()

Customer-product matrix shape: (4336, 3094)


product_name,10002 - inflatable political globe,10080 - groovy cactus inflatable,10120 - doggy rubber,10125 - mini funky design tapes,10133 - colouring pencils brown tube,10135 - colouring pencils brown tube,11001 - asstd design racing car pen,15030 - fan black frame,15034 - paper pocket traveling fan,15036 - assorted colours silk fan,...,90201C - red enamel flower ring,90209B - green enamel+glass hair comb,90209C - pink enamel+glass hair comb,"90214A - letter ""a"" bling key ring","90214K - letter ""k"" bling key ring",BANK CHARGES - bank charges,C2 - carriage,DOT - dotcom postage,M - manual,POST - postage
customer_id,,,,,,,,,,,,,,,,,,,,,
12346,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
12347,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
12348,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
12349,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
12350,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


In [18]:
from sklearn.metrics.pairwise import cosine_similarity

customer_similarity = cosine_similarity(customer_product_matrix)

customer_similarity_df = pd.DataFrame(
    customer_similarity,
    index=customer_product_matrix.index,
    columns=customer_product_matrix.index
)

print("Customer similarity matrix:", customer_similarity_df.shape)

customer_similarity_df.head()

Customer similarity matrix: (4336, 4336)


customer_id,12346,12347,12348,12349,12350,12352,12353,12354,12355,12356,...,18273,18274,18276,18277,18278,18280,18281,18282,18283,18287
customer_id,,,,,,,,,,,,,,,,,,,,,
12346,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000
12347,0.0,1.000000,0.063330,0.046355,0.048029,0.038672,0.0,0.013001,0.137309,0.095205,...,0.0,0.029854,0.052926,0.0,0.033005,0.062622,0.0,0.114332,0.102250,0.012891
12348,0.0,0.063330,1.000000,0.024953,0.051709,0.027756,0.0,0.027995,0.118262,0.146427,...,0.0,0.064282,0.113961,0.0,0.000000,0.000000,0.0,0.000000,0.168363,0.083269
12349,0.0,0.046355,0.024953,1.000000,0.056773,0.121900,0.0,0.030737,0.032461,0.144692,...,0.0,0.105868,0.000000,0.0,0.039014,0.000000,0.0,0.067574,0.113756,0.015237
12350,0.0,0.048029,0.051709,0.056773,1.000000,0.031575,0.0,0.000000,0.000000,0.033315,...,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.044199,0.000000


In [19]:
def recommend_products_collaborative(customer_id, customer_product_matrix, similarity_df, top_n=5):
    customer_id = int(customer_id)

    if customer_id not in customer_product_matrix.index:
        return pd.DataFrame(columns=["recommended_product", "score"])

    similar_customers = (
        similarity_df[customer_id]
        .sort_values(ascending=False)
        .drop(customer_id)
        .head(20)
    )

    similar_customer_ids = similar_customers.index

    similar_customer_products = customer_product_matrix.loc[similar_customer_ids]

    product_scores = similar_customer_products.T.dot(similar_customers)

    already_bought = customer_product_matrix.loc[customer_id]
    product_scores = product_scores[already_bought == 0]

    recommendations = (
        product_scores
        .sort_values(ascending=False)
        .head(top_n)
        .reset_index()
    )

    recommendations.columns = ["recommended_product", "score"]

    return recommendations

In [20]:
sample_customer = customer_product_matrix.index[1]

print("Sample customer:", sample_customer)

recommend_products_collaborative(
    sample_customer,
    customer_product_matrix,
    customer_similarity_df,
    top_n=10
)

Sample customer: 12347


,recommended_product,score
0,23245 - set of 3 regency cake tins,1.535488
1,47566 - party bunting,1.268575
2,23163 - regency sugar tongs,1.213833
3,22730 - alarm clock bakelike ivory,1.110646
4,85123A - white hanging heart t-light holder,1.035169
5,21481 - fawn blue hot water bottle,0.995921
6,22193 - red diner wall clock,0.906635
7,21977 - pack of 60 pink paisley cake cases,0.899916
8,22961 - jam making set printed,0.860904
9,21485 - retrospot heart hot water bottle,0.809786


In [22]:
customer_product_matrix.to_csv(
    "../data/processed/recommendation_customer_product_matrix.csv"
)

customer_similarity_df.to_csv(
    "../data/processed/recommendation_customer_similarity_matrix.csv"
)

print("Collaborative filtering outputs saved successfully.")

Collaborative filtering outputs saved successfully.


In [23]:
recommendation_summary = pd.DataFrame({
    "approach": [
        "Market Basket Analysis / Apriori",
        "Collaborative Filtering"
    ],
    "input_type": [
        "Invoice-level product baskets",
        "Customer-product purchase matrix"
    ],
    "best_for": [
        "Product bundle and cross-sell recommendations",
        "Personalized customer-level recommendations"
    ],
    "strength": [
        "Easy to interpret using support, confidence, and lift",
        "Personalized based on similar customer behavior"
    ],
    "limitation": [
        "Not personalized to individual customers",
        "Can struggle with new customers or sparse purchase history"
    ]
})

recommendation_summary

,approach,input_type,best_for,strength,limitation
0,Market Basket Analysis / Apriori,Invoice-level product baskets,Product bundle and cross-sell recommendations,"Easy to interpret using support, confidence, a...",Not personalized to individual customers
1,Collaborative Filtering,Customer-product purchase matrix,Personalized customer-level recommendations,Personalized based on similar customer behavior,Can struggle with new customers or sparse purc...


In [25]:
recommendation_summary.to_csv(
    "../data/processed/recommendation_approach_comparison.csv",
    index=False
)

print("Recommendation system completed successfully.")

Recommendation system completed successfully.


In [26]:
print("Product Recommendation System Completed Successfully!")

print("\nApproach 1: Market Basket Analysis")
print("Frequent Itemsets:", frequent_itemsets.shape[0])
print("Association Rules:", rules_clean.shape[0])
print("Strong Rules:", strong_rules.shape[0])

print("\nApproach 2: Collaborative Filtering")
print("Customer-Product Matrix:", customer_product_matrix.shape)
print("Customer Similarity Matrix:", customer_similarity_df.shape)

print("\nFiles Saved:")
print("1. data/processed/recommendation_frequent_itemsets.csv")
print("2. data/processed/recommendation_association_rules.csv")
print("3. data/processed/recommendation_strong_rules.csv")
print("4. data/processed/recommendation_customer_product_matrix.csv")
print("5. data/processed/recommendation_customer_similarity_matrix.csv")
print("6. data/processed/recommendation_approach_comparison.csv")

Product Recommendation System Completed Successfully!

Approach 1: Market Basket Analysis
Frequent Itemsets: 972
Association Rules: 934
Strong Rules: 612

Approach 2: Collaborative Filtering
Customer-Product Matrix: (4336, 3094)
Customer Similarity Matrix: (4336, 4336)

Files Saved:
1. data/processed/recommendation_frequent_itemsets.csv
2. data/processed/recommendation_association_rules.csv
3. data/processed/recommendation_strong_rules.csv
4. data/processed/recommendation_customer_product_matrix.csv
5. data/processed/recommendation_customer_similarity_matrix.csv
6. data/processed/recommendation_approach_comparison.csv
